# OmeZarrWriter on the Ti2-E

Runs a short experiment with stimulation and a reference frame and
stores raw images, stimulation readouts, segmentation masks and
tracked labels in a single OME-Zarr v0.5 store. Same experiment as
`tiff_writer.ipynb`.

## 1. Connect to the Ti2-E

`Ti2E()` loads `NikonTi2e.cfg`, applies the `System/Startup` preset and
selects the `Channel` group. Pass `config=r"..."` to load another file.
Close NIS-Elements and any other Micro-Manager first: a device can only be
opened by one program at a time.

In [ ]:
from faro_aoki.ti2e import Ti2E
import faro.core.utils as utils

mic = Ti2E()
utils.print_configs(mic.mmc)

## 2. Settings

Two imaging channels on every frame, blue light through the `GFP` preset on
the stim frames, and a reference channel on the last frame. The positions
start at the current stage position and step `STEP_UM` in x, so the sample
must reach that far. Positions have no `z`, so the focus is not moved.

In [ ]:
import os
import tempfile
from datetime import datetime

IMAGING_CHANNELS = [
    {"config": "iRFP", "exposure": 100},  # nuclear marker, segmented
    {"config": "RFP", "exposure": 100},
]
STIM_CHANNEL = {"config": "GFP", "exposure": 100}  # blue light, whole FOV
REF_CHANNEL = {"config": "YFP_Custom", "exposure": 200}  # last frame only
N_POSITIONS = 3
STEP_UM = 300.0  # x distance between positions
INTERVAL_S = 5.0
N_FRAMES = 6
STIM_FRAMES = [1, 3]

path = os.path.join(
    tempfile.gettempdir(), f"faro-ti2e-omezarr-{datetime.now():%Y%m%d-%H%M%S}"
)
print("results go to", path)

## 3. Set up the pipeline

Otsu segmentation on the first imaging channel, trackpy, `SimpleFE`, and
`StimWholeFOV`: the Ti2-E has no DMD, so stimulation lights the whole
field of view.

In [ ]:
from faro.core.data_structures import SegmentationMethod
from faro.core.pipeline import ImageProcessingPipeline
from faro.feature_extraction.simple import SimpleFE
from faro.segmentation.base import OtsuSegmentator
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=OtsuSegmentator(),
        use_channel=0,
        save_tracked=True,
    )
]

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(),
    stimulator=StimWholeFOV(),
)

## 4. Create the OmeZarrWriter

With `store_stim_images=True` the stimulation readout becomes an
extra channel of the raw array, filled with zeros on frames without
stimulation.

In [ ]:
from faro.core.writers import OmeZarrWriter

writer = OmeZarrWriter(
    storage_path=path,
    raw_chunk_t=1,  # one timepoint per chunk
    label_chunk_t=1,  # random access per frame
    label_shard_t=50,  # 50 frames per shard file
    store_stim_images=True,
)
print(f"Zarr store: {writer._zarr_path}")

## 5. Define the experiment

In [ ]:
from faro.core.data_structures import RTMSequence

x0, y0 = mic.mmc.getXYPosition()
positions = [{"x": x0 + i * STEP_UM, "y": y0} for i in range(N_POSITIONS)]

seq = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_FRAMES},
    stage_positions=positions,
    channels=IMAGING_CHANNELS,
    stim_channels=[STIM_CHANNEL],
    stim_frames=STIM_FRAMES,
    ref_channels=[REF_CHANNEL],
    ref_frames=[-1],
)

events = list(seq)
print(f"{len(events)} events")

## 6. Run the experiment

The stage returns to the first position at the end, so you can run the
notebook again from the same place.

In [ ]:
from faro.core.controller import Controller

ctrl = Controller(mic, pipeline, writer=writer)
assert ctrl.validate_events(events), "fix the warnings above before running"
ctrl.run_experiment(events).wait()
ctrl.finish_experiment()
mic.mmc.setXYPosition(x0, y0)
print("Experiment complete.")

## 7. Inspect the OME-Zarr store

With several positions the raw array is `(t, p, c, y, x)` and each
label array `(t, p, y, x)`. The channels are the imaging channels
followed by the stimulation readout. Reference frames have other
channels, so they go to `ref/` as TIFF files.

In [ ]:
import zarr

store = zarr.open(writer._zarr_path, mode="r")
print(store.tree())

In [ ]:
raw = store["0"]
print(f"Raw array: shape={raw.shape}, dtype={raw.dtype}")
ome = store.attrs.get("ome", {})
axes = ome.get("multiscales", [{}])[0].get("axes", [])
print(f"Axes: {[a['name'] for a in axes]}")
channels = ome.get("omero", {}).get("channels", [])
print(f"Channels: {[ch.get('label') for ch in channels]}")

In [ ]:
labels_grp = store["labels"]
for name in labels_grp.attrs.get("labels", []):
    arr = store[f"labels/{name}/0"]
    print(f"labels/{name}: shape={arr.shape}, dtype={arr.dtype}")

ref_dir = os.path.join(path, "ref")
print("ref/:", sorted(os.listdir(ref_dir)) if os.path.isdir(ref_dir) else "-")

In [ ]:
import matplotlib.pyplot as plt

fov, t_stim, t_last = 0, STIM_FRAMES[0], N_FRAMES - 1
n_img = len(IMAGING_CHANNELS)
panels = [
    (raw[t_last, fov, 0], f"{IMAGING_CHANNELS[0]['config']} (last frame)"),
    (raw[t_last, fov, 1], f"{IMAGING_CHANNELS[1]['config']} (last frame)"),
    (store["labels/labels/0"][t_last, fov], "Labels"),
    (raw[t_stim, fov, n_img], f"Stim readout (frame {t_stim})"),
]
fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
for ax, (img, title) in zip(axes, panels):
    ax.imshow(img, cmap="nipy_spectral" if title == "Labels" else "gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

In [ ]:
import pandas as pd

for fov in range(N_POSITIONS):
    tracks = pd.read_parquet(os.path.join(path, "tracks", f"{fov}_latest.parquet"))
    print(
        f"Position {fov}: {tracks['particle'].nunique()} cells "
        f"across {tracks['timestep'].nunique()} frames"
    )
tracks.head()